In [14]:
# LOAD PACKAGES 
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
opj = os.path.join
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.stats as stats
from prfpy_csenf.stimulus import PRFStimulus2D
from prfpy_csenf.model import Iso2DGaussianModel, Norm_Iso2DGaussianModel
# SCOTOMA PROJECT 
from artscot_JOV.load_saved_info import (
    load_data_prf, get_roi, get_us_factor,
    get_design_matrix_npy, get_scotoma_info, default_prf_dir,
)
from artscot_JOV.prfpy_functions import PrfMulti,Prf1T1M
from artscot_JOV.prfpy_ts_plotter import TSPlotter
from artscot_JOV.stats import cortical_permutation_test
from dpu_mini.stats import dag_get_corr
# Fix seed (just in case doing anything random)
np.random.seed(0)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# OUTPUT DIRECTORY
fig_output = './figures3'
os.makedirs(fig_output, exist_ok=True)
as_cols = {
    'AS0'   : '#000000',
    'AS1'   : '#d95f02',
    'AS2'   : '#7570b3',
}

# Load subjects - sped up by multiprocessing
ecc_bins = np.linspace(0,5,6)
# More IP plotting functions

mod_cols = {
    'gauss' : '#3CA969',
    'G' : '#3CA969',
    'norm' :  '#8557A7',
    'N' :  '#8557A7',
}

## LOAD

#### Where and what to load

In [8]:
task_list = ['AS0', 'AS1', 'AS2']
look_in = default_prf_dir

model_list = ['gauss' , 'norm']
sub_list = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07'] 

roi_fit = 'all'
roi_list = ['all', 'v1custom', 'v2custom', 'v3custom', 'v3abcustom', 'v4custom', 'LOcustom', 'TOcustom', 'IPScustom']



#### Load stimuli

In [9]:
scotoma_info = get_scotoma_info(sub_list[0])
dm_npy = get_design_matrix_npy(['AS0', 'AS1', 'AS2'])
fit_settings = load_data_prf(
    sub_list[0], 'AS0', 'gauss', var_to_load='settings',
    look_in=look_in, 
    )['AS0']['gauss']
prfpy_stim = {}
prfpy_mod = {'gauss':{}, 'norm':{}}
cut_vols = 5
for stim in dm_npy.keys():
    prfpy_stim[stim] = PRFStimulus2D(
        screen_size_cm=fit_settings['screen_size_cm'],
        screen_distance_cm=fit_settings['screen_distance_cm'],
        design_matrix=dm_npy[stim][:,:,cut_vols:], 
        axis=0,
        TR=fit_settings['TR']
        )
    prfpy_mod['gauss'][stim] = Iso2DGaussianModel(prfpy_stim[stim])
    prfpy_mod['norm'][stim] = Norm_Iso2DGaussianModel(prfpy_stim[stim])



task-AS0
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-gauss_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl


In [10]:
def simplify_prf_names(prf_name):
    prf_out = prf_name.replace('gauss', 'G')
    prf_out = prf_out.replace('norm', 'N')
    prf_out = prf_out.replace('_', 'f_') # 
    return prf_out


#### Load the fits

In [11]:
from joblib import Parallel, delayed

# --- Parallel Processing Setup ---
# takes ages to load each subject 
# Define the function that processes a single subject
def process_subject(sub, task_list, look_in, model_list, roi_fit, prfpy_mod):
    """
    Function that encapsulates the original loop body for one subject.
    Returns the key (subject) and the value (PrfMulti object) for the final dictionary.
    """
    
    # 2. Load PRF Parameters
    prf_pars = load_data_prf(
        sub=sub,
        task_list=task_list,
        look_in=look_in,
        model_list=model_list,
        roi_fit=roi_fit,
        include='tc',
    )
    
    # 3. Create TSPlotter objects (Inner loop - Keep sequential for simplicity/overhead)
    prf_obj_list = []
    prf_id_list = []
    for iT, task in enumerate(task_list):
        dm_task = task.split('_')[0]
        for model in model_list:
            prf_obj_list.append(Prf1T1M(
                prf_params = prf_pars[task][model],
                model=model,
            ))
            prf_id_list.append(
                simplify_prf_names(f'{task}_{model}'))

    # 4. Create the PrfMulti object
    subject_prf_multi = PrfMulti(
        prf_obj_list,
        prf_id_list,
    )
    
    return sub, subject_prf_multi

# --- Execute Parallelization ---

print("Starting parallel processing...")
results = Parallel(n_jobs=1, verbose=10)(
    delayed(process_subject)(
        sub, 
        task_list, 
        look_in, 
        model_list, 
        roi_fit, 
        prfpy_mod
    )
    for sub in sub_list
)
print("Parallel processing complete.")

# 5. Reassemble the final dictionary from the results
prf_obj = dict(results)

Starting parallel processing...
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-gauss_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-norm_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-gauss_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-norm_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-gauss_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-01/ses-1/sub-01_model-norm_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
prf_params.shape[-1]=8
include hrf = True
include rsq = True
prf_params

/Users/marcusdaghlian/projects/artscot_JOV/analysis/artscot_JOV/prfpy_functions.py:296: RuntimeWarning: invalid value encountered in divide
  self.params_dd['size_ratio'] = self.params_dd['size_2'] / self.params_dd['size_1']
/Users/marcusdaghlian/projects/artscot_JOV/analysis/artscot_JOV/prfpy_functions.py:297: RuntimeWarning: invalid value encountered in divide
  self.params_dd['amp_ratio'] = self.params_dd['amp_2'] / self.params_dd['amp_1']
/Users/marcusdaghlian/projects/artscot_JOV/analysis/artscot_JOV/prfpy_functions.py:299: RuntimeWarning: invalid value encountered in divide
  self.params_dd['bd_ratio'] = self.params_dd['b_val'] / self.params_dd['d_val']
/Users/marcusdaghlian/projects/artscot_JOV/analysis/artscot_JOV/prfpy_functions.py:301: RuntimeWarning: invalid value encountered in divide
  self.params_dd['sup_idx'] = (self.params_dd['amp_1'] * self.params_dd['size_1']**2) / (self.params_dd['amp_2'] * self.params_dd['size_2']**2)
[Parallel(n_jobs=1)]: Done   1 tasks      | elap

/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-gauss_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-norm_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-gauss_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-norm_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-gauss_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-02/ses-1/sub-02_model-norm_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
prf_params.shape[-1]=8
include hrf = True
include rsq = True
prf_params.shape[-1]=12
include hrf = True

[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    7.6s


/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-gauss_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-norm_roi-all_task-AS0-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-gauss_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-norm_roi-all_task-AS1-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-gauss_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
/Users/marcusdaghlian/projects/artscot_JOV/prf_for_pub/sub-05/ses-1/sub-05_model-norm_roi-all_task-AS2-fits_stage-iter_constr-tc_desc-prf_params.pkl
prf_params.shape[-1]=8
include hrf = True
include rsq = True
prf_params.shape[-1]=12
include hrf = True

[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:   12.9s
[Parallel(n_jobs=1)]: Done   7 out of   7 | elapsed:   12.9s finished


### Get pair wise dataframe containing relevant info

In [15]:

def PAIRED_df(prf_obj, roi_list, p1diff, p2diff, diff_ids,  **kwargs):
    # HOW THRESH...
    # only_close_to_scot = kwargs.get('only_close_to_scot', False)
    min_rsq = kwargs.get('min_rsq', 0.1)
    max_ecc = kwargs.get('max_ecc', 5)
    min_vx = kwargs.get('min_vx', 50)

    sub_list = list(prf_obj.keys())
    nice_roi_list = [i.split('custom')[0] for i in roi_list]

    # Create threshold dictionary    
    th = {}
    for th_key in [*p1diff, *p2diff]:
        th[f'{th_key}-min-rsq'] = min_rsq
        th[f'{th_key}-max-ecc'] = max_ecc    

    # Difference
    diff_stats = {}
    diff_dict = {
        'sub' : [],
        'roi' : [],
        'condition' : [],
        'shift' : [],
    }
    diff_vxwise = {}
    for iS,sub in enumerate(sub_list):
        diff_vxwise[sub] = {}
        diff_stats[sub] = {}
        vx_mask  = prf_obj[sub].return_vx_mask(th)
        for iR, roi in enumerate(roi_list):
            nice_roi = nice_roi_list[iR]
            vx_mask  = prf_obj[sub].return_vx_mask(th)
            vx_mask &= get_roi(sub, roi, combine_matches=True) # ALL                 
            if vx_mask.sum()<min_vx:
                # print(f'failed to get {roi} for {sub}')        
                continue
            shift1s = prf_obj[sub].return_diff_params(
                p1diff[0], p1diff[1], 'shift_mag')['shift_mag'][vx_mask].to_numpy()
            shift1 = np.median(shift1s)
            shift2s = prf_obj[sub].return_diff_params(
                p2diff[0], p2diff[1], 'shift_mag')['shift_mag'][vx_mask].to_numpy()
            shift2 = np.median(shift2s)
            diff_stats[sub][nice_roi] = cortical_permutation_test(
                shift1s, shift2s, subsample_fraction=1/get_us_factor(sub), 
                n_subsamples=10000, alternative='two-sided', random_seed=123
            )
            diff_vxwise[sub][roi] = {
                'vx_mask' : vx_mask, 
                diff_ids[0]: shift1s, 
                diff_ids[1]: shift2s, 
            }
            # Do the stats
            # diff_stats[sub][nice_roi] = dag_paired_ttest(shift1s, shift2s, upsampling_factor=upsampling_factor)
            # shift 1
            diff_dict['sub'].append(sub)
            diff_dict['roi'].append(nice_roi)
            diff_dict['condition'].append(diff_ids[0])
            diff_dict['shift'].append(shift1)
            # shift 2
            diff_dict['sub'].append(sub)
            diff_dict['roi'].append(nice_roi)
            diff_dict['condition'].append(diff_ids[1])
            diff_dict['shift'].append(shift2)
            
    df = pd.DataFrame(diff_dict)
    return df, diff_vxwise, diff_stats


#### Across ROIs

# STATS 

In [16]:
%matplotlib inline

min_rsq = 0.1
min_vx = 50
max_ecc = 5
df_task = {}
dvx_task = {}
for iT,task in enumerate(['AS1', 'AS2']):    
    print('')
    print(f'TASK!! {task}')
    df, df_vxwise, diff_stats = PAIRED_df(
        prf_obj=prf_obj,
        roi_list=roi_list[1::],
        min_rsq=min_rsq,
        max_ecc=max_ecc,
        min_vx = min_vx,
        p1diff=['AS0f_G',    f'{task}f_G'],
        p2diff=['AS0f_N',    f'{task}f_N'],
        diff_ids = ['gauss', 'norm'],
    )
    df_task[task] = df.copy()
    dvx_task[task] = df_vxwise.copy()
 


TASK!! AS1

TASK!! AS2


# [1] Group stats - if asked for 


In [55]:
from statsmodels.stats.multitest import multipletests

for task in df_task.keys():
    df = df_task[task]
    results = []

    for roi in df['roi'].unique():
        roi_data = df[df['roi'] == roi]
        gauss_values = roi_data[roi_data['condition'] == 'gauss']['shift'].values
        norm_values = roi_data[roi_data['condition'] == 'norm']['shift'].values
        
        t_stat, p_value = stats.ttest_rel(gauss_values, norm_values, alternative='two-sided')
        results.append({
            'roi': roi,
            't_statistic': t_stat,
            'p_value': p_value
        })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # --- MULTIPLE COMPARISON CORRECTION ---
    # 'fdr_bh' is the Benjamini-Hochberg procedure (False Discovery Rate)
    # 'bonferroni' is a more conservative alternative
    reject, pvals_corrected, _, _ = multipletests(results_df['p_value'], 
                                                  alpha=0.05, 
                                                  method='fdr_bh')
    
    results_df['p_corrected'] = pvals_corrected
    results_df['significant'] = reject
    results_df.to_csv(f'z_{task}')
    print(f'***********TASK = {task}***********')
    print(results_df)

***********TASK = AS1***********
    roi  t_statistic   p_value  p_corrected  significant
0    v1    -0.464516  0.658656     0.658656        False
1    v2     3.709256  0.009978     0.017320         True
2    v3     2.418531  0.051967     0.069289        False
3  v3ab     3.932914  0.007687     0.017320         True
4    v4     1.997854  0.102219     0.116822        False
5    LO     5.390129  0.001679     0.006718         True
6    TO     6.216947  0.000800     0.006399         True
7   IPS     3.640754  0.010825     0.017320         True
***********TASK = AS2***********
    roi  t_statistic   p_value  p_corrected  significant
0    v1    -2.039189  0.087543     0.116724        False
1    v2     0.574405  0.586568     0.586568        False
2    v3     4.267639  0.005277     0.014071         True
3  v3ab     4.427415  0.004435     0.014071         True
4    v4     1.384528  0.238422     0.272482        False
5    LO     3.016422  0.023505     0.046054         True
6    TO     5.662581  

# [2] Confirm seed does not lead to bias in permutation tests